In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # Force CPU, disables GPU lookup

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Auto-detect dataset path
BASE = '/kaggle/input/datasets/jonathanoheix/face-expression-recognition-dataset'

possible_trains = [
    f'{BASE}/images/train',
    f'{BASE}/train',
    f'{BASE}/images/Train',
    f'{BASE}/Train',
]

TRAIN_DIR = None
for p in possible_trains:
    if os.path.exists(p):
        TRAIN_DIR = p
        break

if TRAIN_DIR:
    VAL_DIR = TRAIN_DIR.replace('train', 'validation').replace('Train', 'validation')
    if not os.path.exists(VAL_DIR):
        VAL_DIR = TRAIN_DIR.replace('train', 'val')
    print(f"Train dir : {TRAIN_DIR}")
    print(f"Val   dir : {VAL_DIR}")
else:
    print("Dataset not found! Add it via + Add Data on the right panel.")

IMG_SIZE    = 48
BATCH_SIZE  = 64
EPOCHS      = 50
NUM_CLASSES = 7
CLASS_NAMES = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

# Class distribution
if TRAIN_DIR:
    for split, path in [('Train', TRAIN_DIR), ('Validation', VAL_DIR)]:
        print(f"\n{split} set:")
        total = 0
        for cls in sorted(os.listdir(path)):
            cls_path = os.path.join(path, cls)
            if os.path.isdir(cls_path):
                count = len(os.listdir(cls_path))
                total += count
                print(f"  {cls:12s}: {count:5d} images")
        print(f"  {'TOTAL':12s}: {total:5d} images")

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("Class indices:", train_generator.class_indices)
print(f"Train batches: {len(train_generator)} | Val batches: {len(val_generator)}")

In [ ]:
def show_samples(generator, class_names):
    images, labels = next(generator)
    
    # Collect 1 sample per class
    collected = {i: None for i in range(len(class_names))}
    for img, lbl in zip(images, labels):
        idx = np.argmax(lbl)
        if collected[idx] is None:
            collected[idx] = img

    fig, axes = plt.subplots(1, len(class_names), figsize=(16, 3))
    for i, ax in enumerate(axes):
        if collected[i] is not None:
            ax.imshow(collected[i].squeeze(), cmap='gray')
        ax.set_title(class_names[i], fontsize=10, fontweight='bold')
        ax.axis('off')

    plt.suptitle("Sample Image per Emotion Class", fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

show_samples(train_generator, CLASS_NAMES)

In [ ]:
train_counts = []
val_counts   = []

for cls in CLASS_NAMES:
    train_counts.append(len(os.listdir(os.path.join(TRAIN_DIR, cls))))
    val_counts.append(len(os.listdir(os.path.join(VAL_DIR, cls))))

x = np.arange(len(CLASS_NAMES))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, train_counts, width, label='Train',      color='steelblue')
bars2 = ax.bar(x + width/2, val_counts,   width, label='Validation', color='coral')

ax.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('Number of Images')
ax.legend()

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
def build_model(input_shape=(48, 48, 1), num_classes=7):
    model = models.Sequential([

        # Block 1 — Basic edges & textures
        layers.Conv2D(32, (3,3), padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(32, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Block 2 — Eye/nose/mouth shapes
        layers.Conv2D(64, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(64, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Block 3 — High-level expression features
        layers.Conv2D(128, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(128, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Classifier Head
        layers.Flatten(),
        layers.Dense(256),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.5),

        layers.Dense(128),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),

        layers.Dense(num_classes, activation='softmax')
    ])
    return model

model = build_model()
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("Model compiled successfully!")

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        'best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]
print("Callbacks ready!")

In [ ]:
print("Starting training...")
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=10,
    validation_data=val_generator,
    validation_steps=len(val_generator),
    callbacks=callbacks,
    verbose=1
)
print("Training complete!")

In [ ]:
def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history['accuracy'],     label='Train', linewidth=2, color='steelblue')
    axes[0].plot(history.history['val_accuracy'], label='Val',   linewidth=2, color='coral')
    axes[0].set_title('Accuracy over Epochs', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history['loss'],     label='Train', linewidth=2, color='steelblue')
    axes[1].plot(history.history['val_loss'], label='Val',   linewidth=2, color='coral')
    axes[1].set_title('Loss over Epochs', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_history(history)

In [ ]:
best_model = keras.models.load_model('best_model.keras')

val_loss, val_acc = best_model.evaluate(val_generator, verbose=1)
print(f"\nFinal Results:")
print(f"  Validation Accuracy : {val_acc*100:.2f}%")
print(f"  Validation Loss     : {val_loss:.4f}")

# Predictions
val_generator.reset()
y_pred_probs = best_model.predict(val_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = val_generator.classes

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100  # % version

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', ax=axes[0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap='Blues')
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

sns.heatmap(cm_pct, annot=True, fmt='.1f', ax=axes[1],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap='Greens')
axes[1].set_title('Confusion Matrix (% per class)', fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()